<img src="https://datascientest.fr/train/assets/logo_liora.png" style="height:150px">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> AI Engineers — Démo Part 4 </h1></center>
<center><h2> Choisir un modèle </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

> **Part 4 · 15 min · ≈ 0,02 $ · prépare le Lab 1**
> Avant la séance : `make corpus` (la section 4 lit F2258).

![](assets/04_tradeoff.png)

In [4]:
import os
import time

# Move to the repository root when Jupyter starts the kernel in the demo folder.
if os.path.basename(os.getcwd()) == "demo":
    os.chdir("../..")

import httpx
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
pd.set_option("display.max_colwidth", 300)

gateway = {"base_url": os.environ["LLM_BASE_URL"], "api_key": os.environ["LLM_API_KEY"],
           "include_response_headers": True}


def measure(name, model, prompt):
    """Appelle un modèle et renvoie la réponse avec ses tokens, sa latence et son coût."""
    start = time.perf_counter()
    response = model.invoke(prompt)
    latency = time.perf_counter() - start
    details = response.usage_metadata.get("output_token_details", {})
    return {
        "modèle": name,
        "tokens entrée": response.usage_metadata["input_tokens"],
        "dont en cache": response.usage_metadata.get("input_token_details", {}).get("cache_read", 0),
        "tokens sortie": response.usage_metadata["output_tokens"],
        "dont raisonnement": details.get("reasoning", 0),
        "latence (s)": round(latency, 2),
        "coût ($)": float(response.response_metadata["headers"]["x-litellm-response-cost"]),
        "réponse": response.text.strip(),
    }


print("Prêt")

Prêt


<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 1. Le catalogue de la gateway </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Les fiches techniques des modèles, **lues en direct** dans la gateway : contexte, prix, raisonnement.

In [5]:
response = httpx.get(os.environ["LLM_BASE_URL"].rstrip("/") + "/v1/model/info",
                    headers={"Authorization": "Bearer " + os.environ["LLM_API_KEY"]})

selected = ["gpt-5.4-nano", "gpt-4o-mini", "gpt-5.4-mini", "gpt-5.4",
           "anthropic/claude-sonnet-4-6", "anthropic/claude-opus-4-6"]
rows = []
for model in response.json()["data"]:
    if model["model_name"] in selected:
        model_info = model["model_info"]
        rows.append({
            "modèle": model["model_name"],
            "contexte max (tokens)": model_info["max_input_tokens"],
            "réponse max (tokens)": model_info["max_output_tokens"],
            "$ / M tokens entrée": (model_info["input_cost_per_token"] or 0) * 1_000_000,
            "$ / M tokens sortie": (model_info["output_cost_per_token"] or 0) * 1_000_000,
            "raisonnement": bool(model_info.get("supports_reasoning")),
        })

pd.DataFrame(rows).sort_values("$ / M tokens sortie")

,modèle,contexte max (tokens),réponse max (tokens),$ / M tokens entrée,$ / M tokens sortie,raisonnement
3,gpt-4o-mini,128000,16384,0.15,0.60,False
1,gpt-5.4-nano,272000,128000,0.20,1.25,True
0,gpt-5.4-mini,272000,128000,0.75,4.50,True
2,gpt-5.4,1050000,128000,2.50,15.00,True
5,anthropic/claude-sonnet-4-6,1000000,128000,3.00,15.00,True
4,anthropic/claude-opus-4-6,1000000,128000,5.00,25.00,True


- Prix de sortie : **0,60 $ → 25 $** par million de tokens (× 40).
- **La sortie coûte 4 à 6 × l'entrée.**
- Contexte : **128 k → 1 M tokens**. Nos 55 fiches ≈ 100 k : elles tiennent… → **pourquoi un RAG ?** (Part 5)

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 2. Même question, trois modèles </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- **Même question**, trois tailles de modèle.
- Lire les réponses **avant** de montrer les prix.

In [7]:
QUESTION = ("Un salarié demande : « Mon employeur peut-il m'obliger à prendre mes congés en août ? » "
            "Répondez en trois phrases au plus, pour un salarié du secteur privé en France.")

three_models = [
    ("gpt-5.4-nano", ChatOpenAI(model="gpt-5.4-nano", temperature=0, **gateway)),
    ("gpt-4o-mini", ChatOpenAI(model="gpt-4o-mini", temperature=0, **gateway)),
    ("claude-sonnet-4-6", ChatOpenAI(model="anthropic/claude-sonnet-4-6", temperature=0, **gateway)),
]

rows = []
for name, model in three_models:
    rows.append(measure(name, model, QUESTION))
    
pd.DataFrame(rows)#.drop(columns=["dont en cache", "dont raisonnement", "réponse"])

,modèle,tokens entrée,dont en cache,tokens sortie,dont raisonnement,latence (s),coût ($),réponse
0,gpt-5.4-nano,47,0,122,0,1.31,0.000162,"En France, l’employeur peut fixer les dates de congés payés, mais uniquement dans le respect des règles prévues par le Code du travail, la convention collective et l’accord d’entreprise (et en tenant compte des contraintes de service). En général, l’employeur doit consulter les salariés/tenir co..."
1,gpt-4o-mini,48,0,88,0,1.62,0.000060,"En France, l'employeur peut imposer des dates de congés payés, y compris en août, mais cela doit être fait dans le respect des règles prévues par le Code du travail et les conventions collectives. Il doit également consulter le salarié et tenir compte de ses souhaits dans la mesure du possible. ..."
2,claude-sonnet-4-6,58,0,169,0,4.29,0.002709,"En France, l'employeur a le pouvoir de fixer les dates de congés payés, sous réserve de respecter les règles légales et conventionnelles (convention collective, accord d'entreprise). Il doit vous informer au moins **un mois à l'avance** de vos dates de congés et ne peut pas les imposer sans resp..."


In [8]:
for row in rows:
    print(row["modèle"], ":")
    print(row["réponse"])
    print()

gpt-5.4-nano :
En France, l’employeur peut fixer les dates de congés payés, mais uniquement dans le respect des règles prévues par le Code du travail, la convention collective et l’accord d’entreprise (et en tenant compte des contraintes de service). En général, l’employeur doit consulter les salariés/tenir compte des souhaits dans la mesure du possible et respecter des délais de fixation. Ainsi, il peut vous obliger à prendre des congés en août, sauf si cela heurte ces règles (ex. priorité des dates, délais, ou restrictions prévues par votre accord collectif).

gpt-4o-mini :
En France, l'employeur peut imposer des dates de congés payés, y compris en août, mais cela doit être fait dans le respect des règles prévues par le Code du travail et les conventions collectives. Il doit également consulter le salarié et tenir compte de ses souhaits dans la mesure du possible. Toutefois, si des périodes de congés sont fixées par l'entreprise, le salarié doit s'y conformer.

claude-sonnet-4-6 :
En

- Les trois disent l'essentiel ; les écarts portent sur **la précision et le ton**.
- La règle de la fiche : **1er mai – 31 octobre**, 12 jours ouvrables de suite. **Qui la cite ? Comment vérifier sans source ?** → le RAG.
- Claude Sonnet : **≈ 40 à 50 × plus cher** que gpt-4o-mini, ≈ 3 × plus lent.
- Pour trancher : **une grille et un jeu de questions** (Lab 5).

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 3. La température </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- La température = **la dose de hasard** dans le choix de chaque token.

```
T = 0    →  toujours le token le plus probable  →  réponses identiques
T = 1,2  →  des tokens moins probables aussi    →  réponses variées
```

In [9]:
PROMPT = "Proposez un nom pour une application mobile de gestion des congés. Un seul mot, sans explication."

for temperature in [0, 1.2]:
    model = ChatOpenAI(model="gpt-4o-mini", temperature=temperature, **gateway)
    names = []
    for trial in range(5):
        names.append(model.invoke(PROMPT).text.strip())
    print("température", temperature, ":", names)

température 0 : ['Congéa', 'Congéa', 'Congéa', 'Congéa', 'Congéa']
température 1.2 : ['Congéa', 'Congés.', 'Congésia', 'Congéos', 'Congeez']


- **T basse** : extraire, classer, répondre à partir de documents.
- **T haute** : proposer des idées, reformuler.
- Même à 0, la reproductibilité **n'est pas garantie** par tous les fournisseurs.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 4. Le prix du contexte </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Chaque token d'entrée est **payé à chaque appel**.
- Une fiche entière en contexte : **≈ 6 000 tokens**.

In [10]:
if not os.path.exists("data/corpus/F2258.md"):
    raise SystemExit("Corpus absent : lancez make corpus depuis la racine du dépôt.")
    
sheet = open("data/corpus/F2258.md", encoding="utf-8").read()
QUESTION_SHEET = "Combien de jours de congés payés un salarié acquiert-il par mois de travail ?"

mini = ChatOpenAI(model="gpt-4o-mini", temperature=0, **gateway)
sonnet = ChatOpenAI(model="anthropic/claude-sonnet-4-6", temperature=0, **gateway)

rows = [
    measure("gpt-4o-mini, question seule", mini, QUESTION_SHEET),
    measure("gpt-4o-mini, question + fiche", mini, sheet + "\n\n" + QUESTION_SHEET),
    measure("claude-sonnet-4-6, question + fiche", sonnet, sheet + "\n\n" + QUESTION_SHEET),
]
pd.DataFrame(rows).drop(columns=["dont raisonnement", "réponse"])

,modèle,tokens entrée,dont en cache,tokens sortie,latence (s),coût ($)
0,"gpt-4o-mini, question seule",26,0,89,1.40,0.000057
1,"gpt-4o-mini, question + fiche",6043,0,28,1.27,0.000923
2,"claude-sonnet-4-6, question + fiche",7885,0,178,4.75,0.026325


- Avec la fiche, **le coût explose**, alors que la réponse est plus courte.
- `dont en cache` : relancée dans les minutes qui suivent, la fiche est **relue à moitié prix** (Part 12).
- **Lire est rapide, écrire est lent** : la latence bouge peu.
- Claude compte **≈ 8 000 tokens** pour le même texte : autre tokenizer.
- **L'argument du RAG** : envoyer 5 passages, pas la fiche ni le corpus.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 5. Le raisonnement </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- **Bonne réponse : 9** (12 jours en août + 4 en décembre = 16 ; 25 − 16 = 9).
- Sans brouillon, le modèle donne **un nombre au jugé**.

In [11]:
PROBLEM = (
    "Un salarié a 25 jours ouvrables de congés payés. Il pose ses congés du lundi 2 août au samedi "
    "14 août 2027 inclus, sans jour férié sur la période. Les jours ouvrables vont du lundi au samedi. "
    "Il pose ensuite 4 jours ouvrables en décembre. Combien de jours ouvrables lui reste-t-il ? "
    "Répondez uniquement par un nombre."
)

configurations = [
    ("gpt-4o-mini", ChatOpenAI(model="gpt-4o-mini", temperature=0, **gateway)),
    ("gpt-5.4-mini sans raisonnement", ChatOpenAI(model="gpt-5.4-mini", temperature=0, reasoning_effort="none", **gateway)),
    ("gpt-5.4-mini avec raisonnement", ChatOpenAI(model="gpt-5.4-mini", reasoning_effort="medium", **gateway)),
]

rows = []
for name, model in configurations:
    rows.append(measure(name, model, PROBLEM))
pd.DataFrame(rows)

,modèle,tokens entrée,dont en cache,tokens sortie,dont raisonnement,latence (s),coût ($),réponse
0,gpt-4o-mini,90,0,1,0,0.60,0.000014,21
1,gpt-5.4-mini sans raisonnement,89,0,10,0,0.62,0.000112,13
2,gpt-5.4-mini avec raisonnement,89,0,205,195,2.05,0.000989,9


- Sans raisonnement : **faux**, pour les deux modèles.
- Avec : **9**, pour **une à plusieurs centaines de tokens** de brouillon et quelques secondes.
- Rentable sur **les calculs enchaînés**, inutile sur une question simple.

**Lab 1 (facultatif)** : même expérience sur une autre question.